# OmniVoice — thử voice cloning tiếng Việt trên Google Colab

Notebook chạy model chính thức [`k2-fsa/OmniVoice`](https://huggingface.co/k2-fsa/OmniVoice) qua Python package `omnivoice`. OmniVoice là model zero-shot TTS đa ngôn ngữ, hỗ trợ clone giọng từ audio tham chiếu.

Trước khi chạy: vào **Runtime → Change runtime type → T4 GPU**. Chỉ dùng audio khi bạn có quyền sử dụng giọng nói đó.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'Hãy chuyển runtime sang GPU rồi chạy lại cell này.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
# Bản PyPI ổn định của OmniVoice và package ghi WAV.
# Không ép torch để giữ bộ CUDA/PyTorch đi kèm Colab.
!pip -q install -U omnivoice soundfile

In [ ]:
from google.colab import files

uploaded = files.upload()
assert uploaded, 'Hãy upload một file WAV/MP3/M4A có một người nói.'
ref_audio = next(iter(uploaded))
print('Reference audio:', ref_audio)

# Mẫu sạch khoảng 3–10 giây, một người nói, ít nhạc/nhiễu/reverb thường cho kết quả tốt hơn.

In [ ]:
# Khuyến nghị điền transcript chính xác. Để None, OmniVoice sẽ tự tải/chạy Whisper để chép lời,
# khiến lần chạy đầu lâu hơn và tốn bộ nhớ hơn.
ref_text = 'Thay câu này bằng transcript CHÍNH XÁC của file audio bạn vừa upload.'
text_to_speak = (
    'Xin chào! Đây là bản thử nghiệm OmniVoice, mô hình chuyển văn bản thành giọng nói '
    'đa ngôn ngữ với khả năng clone giọng zero-shot.'
)

assert 'Thay câu này' not in ref_text, 'Hãy điền transcript cho ref_text trước khi chạy.'

In [ ]:
import torch
from omnivoice import OmniVoice

MODEL_ID = 'k2-fsa/OmniVoice'
print(f'Loading {MODEL_ID}...')
model = OmniVoice.from_pretrained(
    MODEL_ID,
    device_map='cuda:0',
    dtype=torch.float16,
)

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

# output là danh sách waveform 24 kHz.
audio = model.generate(
    text=text_to_speak,
    ref_audio=ref_audio,
    ref_text=ref_text,
)

sample_rate = 24000
output_path = 'omnivoice_output.wav'
sf.write(output_path, audio[0], sample_rate)
print(f'Đã lưu {output_path} ({sample_rate} Hz)')
display(Audio(audio[0], rate=sample_rate))

In [ ]:
from google.colab import files
files.download('omnivoice_output.wav')

## Lưu ý khi benchmark

- Dùng cùng file tham chiếu, transcript và câu test khi so sánh OmniVoice với Gwen-TTS/VieNeu.
- Viết số, ngày tháng, tiền tệ và từ viết tắt theo cách đọc đầy đủ để phát âm tiếng Việt ổn định hơn.
- Với script dài, tách thành các câu/đoạn ngắn trước khi tạo audio.
- Model tự nhận tiếng Việt từ text; không cần truyền mã ngôn ngữ trong luồng clone giọng cơ bản này.